# Chunking strategy

Articles are split by HTML structure, not by length. There is no size limit and no overlap.

1. **Parse** (`DataProcessor.parse_html`): keep the text of `h1`–`h6`, `p`, `li`, `tr`, `blockquote`, and `pre` as ordered `(tag, text)` blocks. Skip `script` and `style`.
2. **Chunk** (`DataProcessor.chunk_blocks`): one chunk per block, except:
   - A heading is merged into the body block after it as `"Heading: body"`. A heading with no body after it becomes its own chunk.
   - Consecutive list items (`li`) in the same section are merged with `"; "` into one chunk, up to 500 characters.
   - A table row (`tr`) with no heading merged in gets its section prefixed, `"Section: row"`. Cells are joined with `": "`.
   - `chunk_id` is `{slug}_{index}`. `index` counts headings too, so IDs are unique but not contiguous.
3. **Save** (`DataProcessor.chunk_all_articles`): joins each chunk to its `plant_care` row on `slug`, so every chunk carries the plant metadata (name, family, care fields, pet safety, and so on). `summary`, `volume_path`, `ingestion_timestamp`, `published_at` and `updated_at` are left out. Writes `plant_care_chunks` when `save_table=True`, overwriting the table and its schema.

**Known structure loss**
- **Tables:** each `tr` is one chunk (153 tags, 153 chunks). Column headers are not tied to their cells.
- **Lists:** items are merged up to 500 characters, so a long list is split across several chunks.
- **Links:** `href` is dropped; the link text is kept.
- **Heading levels:** `section` keeps only the nearest heading, not its parent.
- **Untagged text:** 16 `text` chunks sit outside any block tag. 14 are article introductions and 2 are "Care at a Glance" banners. Both are kept.

In [1]:
import re
from collections import Counter

from loguru import logger
from plant_care.config import get_env, load_config
from plant_care.data_processor import DataProcessor
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.getOrCreate()

# Load config
env = get_env()
cfg = load_config("../project_config.yml", env)

processor = DataProcessor(spark, cfg)

In [2]:
# Optionally, chunk parsed articles without saving to test the strategy
chunks_df = processor.chunk_all_articles(save_table=True)
logger.info(f"Chunks: {chunks_df.count()}")

2026-11-18 10:18:18.142 | INFO     | plant_care.data_processor:chunk_all_articles:228 - Created 1644 chunks from 118 articles
2026-11-18 10:18:21.702 | INFO     | plant_care.data_processor:_save:122 - Saved to llmops_dev.srinidhi_ilango98.plant_care_chunks
2026-11-18 10:18:23.224 | INFO     | __main__:<module>:3 - Chunks: 1644


In [3]:
display(chunks_df.limit(5))

,slug,chunk_id,url,section,text,plant_id,common_name,scientific_name,family,category,tags,difficulty,placement,watering,soil,toxicity,light_description,sunlight_hours,light_intensity,temperature_recommended,humidity_recommended,soil_ph_recommended,fertilizer_frequency,growth_rate,mature_height,pet_safe_cat,pet_safe_dog
0,african-violet,african-violet_1,https://www.plantsolve.com/plants/african-violet/,Overview,"Overview: The African Violet (Saintpaulia) is one of the most beloved indoor flowering plants in the world, prized for its ability to produce vibrant clusters of purple, pink, or white flowers year-round. However, they also have a reputation for being finicky, largely because they are highly susceptible to crown rot and leaf spotting if watered incorrectly. The key to mastering African Violet care is understanding that their fuzzy leaves are a delicate defense mechanism that cannot get wet. This guide will teach you the art of bottom-watering and how to find the perfect light balance to keep your plant blooming continuously.",plant-african-violet,African Violet,Streptocarpus sect. Saintpaulia,Gesneriaceae,Indoor Plants,"indoor-plants, pet-safe, flowering-plants, small-plants, moderate-care",intermediate,Bright indirect light on an East or North facing windowsill.,Bottom-water only when the top inch of soil is dry. Never get water on the fuzzy leaves.,"A very light, airy, peat-free mix (or a dedicated African Violet mix).",non-toxic,"Requires bright, indirect light to bloom continuously. Direct sun will burn the leaves, but too little light will stop flower production completely.",4-6 hours indirect,moderate,65°F - 80°F (18°C - 27°C),50% - 70%,6.0 - 6.5,Every two weeks during spring and summer (or quarter strength at every watering).,moderate,6-12 inches indoors,True,True
1,african-violet,african-violet_2,https://www.plantsolve.com/plants/african-violet/,Overview,"Native to the humid cloud forests of Tanzania and Kenya, African Violets grow in rocky crevices or on the forest floor, sheltered from direct sun and heavy downpours. They absorb moisture from the humid air and the damp, mossy ground. Their leaves are covered in tiny hairs (trichomes) designed to collect dew. When we water them from above with cold tap water in a dry home, we bypass their natural defenses, leading to shock, fungal infections, and rapid death.",plant-african-violet,African Violet,Streptocarpus sect. Saintpaulia,Gesneriaceae,Indoor Plants,"indoor-plants, pet-safe, flowering-plants, small-plants, moderate-care",intermediate,Bright indirect light on an East or North facing windowsill.,Bottom-water only when the top inch of soil is dry. Never get water on the fuzzy leaves.,"A very light, airy, peat-free mix (or a dedicated African Violet mix).",non-toxic,"Requires bright, indirect light to bloom continuously. Direct sun will burn the leaves, but too little light will stop flower production completely.",4-6 hours indirect,moderate,65°F - 80°F (18°C - 27°C),50% - 70%,6.0 - 6.5,Every two weeks during spring and summer (or quarter strength at every watering).,moderate,6-12 inches indoors,True,True
2,african-violet,african-violet_4,https://www.plantsolve.com/plants/african-violet/,Light,"Light: Light is the absolute critical factor for getting an African Violet to bloom. They require bright, indirect light. An east-facing window is ideal because it provides gentle morning sun and bright ambient light for the rest of the day. If the light is too low, the plant will survive and grow green leaves, but it will never produce flowers. Conversely, if placed in harsh, direct afternoon sun (like a south-facing window), the leaves will become pale, scorched, and burnt.",plant-african-violet,African Violet,Streptocarpus sect. Saintpaulia,Gesneriaceae,Indoor Plants,"indoor-plants, pet-safe, flowering-plants, small-plants, moderate-care",intermediate,Bright indirect light on an East or North facing windowsill.,Bottom-water only when the top inch of soil is dry. Never

### Examples of HTML Elements and Chunks

In [4]:
# Verify the tags that occur in the articles to understand structural loss
tags = ["ul", "ol", "li", "table", "tr", "td", "th", "a"]
rows = spark.table(processor.plant_table).filter("volume_path IS NOT NULL").select("volume_path").collect()

tag_total = Counter()
tag_articles = Counter()
for row in rows:
    with open(row["volume_path"], encoding="utf-8") as f:
        html = f.read()
    for t in tags:
        n = len(re.findall(rf"<{t}[\s>/]", html))
        tag_total[t] += n
        if n:
            tag_articles[t] += 1

print(f"Articles read: {len(rows)}")
for t in tags:
    print(f"<{t}>: {tag_total[t]} occurrences in {tag_articles[t]} articles")

spark.table(processor.parsed_table).select(F.explode("blocks").alias("b")).groupBy(F.col("b.tag").alias("tag")).count().orderBy("tag").show()

Articles read: 118
<ul>: 56 occurrences in 20 articles
<ol>: 21 occurrences in 16 articles
<li>: 347 occurrences in 21 articles
<table>: 23 occurrences in 12 articles
<tr>: 153 occurrences in 12 articles
<td>: 506 occurrences in 12 articles
<th>: 89 occurrences in 12 articles
<a>: 195 occurrences in 20 articles
+----+-----+
| tag|count|
+----+-----+
|  h2|  623|
|  h3|  469|
|  li|  347|
|   p| 1357|
|text|   16|
|  tr|  153|
+----+-----+



In [5]:
sizes = chunks_df.select("chunk_id", "slug", F.length("text").alias("chars"))
sizes.select(F.min("chars").alias("min_chars"), F.max("chars").alias("max_chars"), F.avg("chars").alias("avg_chars")).show()

print("Smallest chunks:")
sizes.orderBy("chars").limit(5).show(truncate=False)
print("Largest chunks:")
sizes.orderBy(F.desc("chars")).limit(5).show(truncate=False)

+---------+---------+-----------------+
|min_chars|max_chars|        avg_chars|
+---------+---------+-----------------+
|       30|     1210|354.3625304136253|
+---------+---------+-----------------+

Smallest chunks:
+----------------+-------------+-----+
|chunk_id        |slug         |chars|
+----------------+-------------+-----+
|java-fern_10    |java-fern    |30   |
|zz-plant_147    |zz-plant     |35   |
|bucephalandra_27|bucephalandra|40   |
|bucephalandra_28|bucephalandra|40   |
|java-fern_38    |java-fern    |42   |
+----------------+-------------+-----+

Largest chunks:
+---------------------------+-------------------------+-----+
|chunk_id                   |slug                     |chars|
+---------------------------+-------------------------+-----+
|zz-plant_87                |zz-plant                 |1210 |
|tradescantia-zebrina_2     |tradescantia-zebrina     |1109 |
|maranta-prayer-plant-care_2|maranta-prayer-plant-care|1009 |
|pilea-chinese-money-plant_8|pilea-chinese

In [6]:
# Count chunks per length bucket, so very short chunks stand out
bucket = (
    F.when(F.col("chars") < 50, "a: <50")
    .when(F.col("chars") < 100, "b: 50-99")
    .when(F.col("chars") < 200, "c: 100-199")
    .when(F.col("chars") < 400, "d: 200-399")
    .when(F.col("chars") < 800, "e: 400-799")
    .otherwise("f: 800+")
)
sizes.withColumn("bucket", bucket).groupBy("bucket").count().orderBy("bucket").show()

+----------+-----+
|    bucket|count|
+----------+-----+
|    a: <50|   14|
|  b: 50-99|   56|
|c: 100-199|  213|
|d: 200-399|  740|
|e: 400-799|  595|
|   f: 800+|   26|
+----------+-----+



In [7]:
chunk = chunks_df.filter(F.col("chunk_id") == "bucephalandra_27").collect()[0]
volume_path = spark.table(processor.plant_table).filter(F.col("slug") == chunk["slug"]).select("volume_path").collect()[0]["volume_path"]
with open(volume_path, encoding="utf-8") as f:
    html = f.read()


def cell_texts(row):
    cells = re.findall(r"<t[dh]\b[^>]*>(.*?)</t[dh]>", row, flags=re.S)
    return [" ".join(re.sub(r"<[^>]+>", "", cell).split()) for cell in cells]


raw_rows = re.findall(r"<tr\b.*?</tr>", html, flags=re.S)
match = next(r for r in raw_rows if chunk["text"].endswith(": ".join(cell_texts(r))))
print("RAW HTML:", " ".join(match.split()))
print(f"CHUNK {chunk['chunk_id']} [{chunk['tags']}] section={chunk['section']!r}: {chunk['text']}")

RAW HTML: <tr><td>GH</td><td>3 to 6</td></tr>
CHUNK bucephalandra_27 [] section='Water Chemistry and Lighting': Water Chemistry and Lighting: GH: 3 to 6


In [8]:
articles = spark.table(processor.plant_table).filter("volume_path IS NOT NULL").select("slug", "volume_path").collect()


def first_article_matching(pattern):
    for row in articles:
        with open(row["volume_path"], encoding="utf-8") as f:
            html = f.read()
        m = re.search(pattern, html, flags=re.S)
        if m:
            return row["slug"], re.sub(r"\s+", " ", html[m.start() : m.start() + 300]), m
    return None, None, None


def show_example(title, pattern, chunk_filter):
    slug, snippet, m = first_article_matching(pattern)
    print(f"=== {title} ({slug})")
    print("HTML:", snippet)
    for c in chunks_df.filter(F.col("slug") == slug).filter(chunk_filter(m)).orderBy("chunk_id").limit(2).collect():
        print(f"CHUNK {c['chunk_id']} section={c['section']!r}: text={c['text'][:200]}")


# Show the raw HTML for each structural loss next to the chunks from the same article
show_example("List", r"<li\b.*?</li>", lambda m: F.col("text").contains(re.sub(r"<[^>]+>", "", m.group(0))[:40]))
show_example("Link", r"<a\s[^>]*>([^<]+)</a>", lambda m: F.col("text").contains(m.group(1)))
show_example("Heading level", r"<h3>([^<]+)</h3>", lambda m: F.col("section") == m.group(1))

=== List (anubias-nana)
HTML: <li><strong>Light:</strong> Low to moderate. Standard aquarium LEDs are enough. Check your setup with the <a href="/calculators/sunlight/anubias-nana/">Anubias Nana sunlight calculator</a>.</li><li><strong>Planting:</strong> Attach the rhizome to rock or driftwood. Never bury it.</li><li><strong>Wat
CHUNK anubias-nana_1 section='Anubias Nana Care at a Glance': text=Anubias Nana Care at a Glance: Light: Low to moderate. Standard aquarium LEDs are enough. Check your setup with the Anubias Nana sunlight calculator.; Planting: Attach the rhizome to rock or driftwood
=== Link (aloe-vera)
HTML: <a href="/calculators/potting-mix-calculator/aloe-vera/">fast-draining succulent or cactus growing medium</a> that drains rapidly. Containers must have drainage holes, and the plant should never sit in collected water. A heavy, moisture-retentive mix can remain wet around the roots long after the su
CHUNK aloe-vera_7 section='Soil and Drainage': text=Soil and Drainage: Us

In [9]:
# Chunks with no section: article introductions
print("=== Chunks with no section")
for c in chunks_df.filter(F.col("section") == "").orderBy("chunk_id").limit(3).collect():
    print(f"CHUNK {c['chunk_id']} section={c['section']!r}: {c['text'][:200]}")

=== Chunks with no section
CHUNK alocasia-black-velvet_0 section='': The Alocasia reginula is the undisputed crown jewel of moody indoor foliage, but watching its thick, velvety black leaves curl, crisp, and drop within weeks of bringing it home from the nursery is a f
CHUNK alocasia-frydek_0 section='': Alocasia Frydek, sometimes sold as the Green Velvet Alocasia, is one of the most dramatic foliage plants you can grow indoors: deep, near-black-green arrow-shaped leaves with a soft velvet texture and
CHUNK anthurium-clarinervium_0 section='': Anthurium Clarinervium is the more compact, slightly more forgiving sibling of Anthurium Crystallinum, native to the limestone slopes of Chiapas, Mexico. Its dark, velvety heart-shaped leaves are etch
